In [1]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 zip 파일을 코랩 임시 디스크로 압축 해제 (경로는 본인 파일명에 맞게 수정하세요)
!unzip -q /content/drive/MyDrive/data.zip -d /content/dataset/

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

# GPU(CUDA) 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# --- 하이퍼파라미터 설정 ---
BATCH_SIZE = 64
IMG_SIZE = 224
DATA_DIR = '/content/dataset/synthetic_chart_images' # 압축 푼 데이터 폴더 경로

# --- 1채널(Grayscale) 전처리 파이프라인 ---
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 전체 데이터셋 로드
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# ==========================================
# ⭐️ 핵심 변경: 클래스 비율을 유지하는 층화 추출(Stratified Split)
# ==========================================
# 1. 전체 데이터의 인덱스와 정답 라벨 추출
indices = np.arange(len(full_dataset))
targets = full_dataset.targets

# 2. 1차 분할: Train(70%) / 임시(30%)로 나누기
# (stratify=targets 옵션이 클래스별 비율을 똑같이 맞춰줍니다)
train_idx, temp_idx, _, temp_targets = train_test_split(
    indices, targets, test_size=0.3, stratify=targets, random_state=42
)

# 3. 2차 분할: 남은 임시(30%)를 반반 나눠서 Val(15%) / Test(15%)로 나누기
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=temp_targets, random_state=42
)

# 4. 추출된 인덱스를 바탕으로 PyTorch Subset(부분 데이터셋) 생성
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

# ==========================================

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"총 데이터 수: {len(full_dataset)}장 (분류 클래스: {len(full_dataset.classes)}개)")
print(f"📊 학습(Train): {len(train_dataset)}장 (70%)")
print(f"📊 검증(Val): {len(val_dataset)}장 (15%)")
print(f"📊 테스트(Test): {len(test_dataset)}장 (15%)")
print("✅ 모든 클래스가 정확히 70:15:15 비율로 분배되었습니다!")

총 데이터 수: 110000장 (분류 클래스: 21개)
📊 학습(Train): 77000장 (70%)
📊 검증(Val): 16500장 (15%)
📊 테스트(Test): 16500장 (15%)
✅ 모든 클래스가 정확히 70:15:15 비율로 분배되었습니다!


<h1>데이터 잘 분할되었는지 확인하기</h1>

In [7]:
from collections import Counter
import pandas as pd

def get_class_distribution(subset, full_dataset):
    """Subset 내의 클래스별 데이터 개수를 딕셔너리로 반환"""
    # Subset에 포함된 인덱스의 라벨 추출
    targets = [full_dataset.targets[i] for i in subset.indices]
    counts = Counter(targets)

    # 클래스 인덱스를 클래스 이름으로 매핑
    return {full_dataset.classes[idx]: counts.get(idx, 0) for idx in range(len(full_dataset.classes))}

# 1. 각 분할별 클래스 분포 계산
train_dist = get_class_distribution(train_dataset, full_dataset)
val_dist = get_class_distribution(val_dataset, full_dataset)
test_dist = get_class_distribution(test_dataset, full_dataset)

# 2. Pandas DataFrame으로 깔끔하게 표 형태로 정리
df_dist = pd.DataFrame({
    'Train': train_dist,
    'Val': val_dist,
    'Test': test_dist
})

# 총합 및 비율(%) 컬럼 추가
df_dist['Total'] = df_dist.sum(axis=1)
df_dist['Train (%)'] = (df_dist['Train'] / df_dist['Total'] * 100).round(1)
df_dist['Val (%)'] = (df_dist['Val'] / df_dist['Total'] * 100).round(1)
df_dist['Test (%)'] = (df_dist['Test'] / df_dist['Total'] * 100).round(1)

print("📊 [클래스별 데이터 분할 현황]")
print(df_dist)

📊 [클래스별 데이터 분할 현황]
          Train   Val  Test  Total  Train (%)  Val (%)  Test (%)
class_0    3500   750   750   5000       70.0     15.0      15.0
class_1    3500   750   750   5000       70.0     15.0      15.0
class_10   3500   750   750   5000       70.0     15.0      15.0
class_11   3500   750   750   5000       70.0     15.0      15.0
class_12   3500   750   750   5000       70.0     15.0      15.0
class_13   3500   750   750   5000       70.0     15.0      15.0
class_14   3500   750   750   5000       70.0     15.0      15.0
class_15   3500   750   750   5000       70.0     15.0      15.0
class_16   3500   750   750   5000       70.0     15.0      15.0
class_17   3500   750   750   5000       70.0     15.0      15.0
class_18   3500   750   750   5000       70.0     15.0      15.0
class_19   3500   750   750   5000       70.0     15.0      15.0
class_2    3500   750   750   5000       70.0     15.0      15.0
class_20   7000  1500  1500  10000       70.0     15.0      15.0
class_

In [8]:
NUM_CLASSES = 21 # 패턴 20개 + Other 1개

# 1. 사전 학습된 ResNet-18 불러오기
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. 첫 번째 레이어 1채널 개조
original_conv1 = model.conv1
model.conv1 = nn.Conv2d(in_channels=1,
                        out_channels=original_conv1.out_channels,
                        kernel_size=original_conv1.kernel_size,
                        stride=original_conv1.stride,
                        padding=original_conv1.padding,
                        bias=False)

# 3채널 가중치를 평균 내어 1채널에 이식
with torch.no_grad():
    model.conv1.weight[:] = original_conv1.weight.sum(dim=1, keepdim=True) / 3.0

# 3. 마지막 분류기(fc) 21개로 교체
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, NUM_CLASSES)

model = model.to(device)
print("ResNet-18 (1채널 개조 & 21개 클래스) 세팅 완료!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 243MB/s]


ResNet-18 (1채널 개조 & 21개 클래스) 세팅 완료!


In [9]:
EPOCHS = 30           # 최대 30 에폭
LEARNING_RATE = 0.001
PATIENCE = 5          # 5번 연속으로 검증 성능이 안 오르면 조기 종료
BEST_MODEL_PATH = 'best_chart_resnet_model.pth' # 최고 성능 모델 저장 경로

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 얼리 스토핑용 변수 초기화
best_val_loss = float('inf')
patience_counter = 0

print("🚀 학습 및 얼리 스토핑 추적을 시작합니다...")

for epoch in range(EPOCHS):
    # --- 1. Training (학습) ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100. * train_correct / train_total

    # --- 2. Validation (검증) ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100. * val_correct / val_total

    print(f"[Epoch {epoch+1}/{EPOCHS}] "
          f"Train Loss: {avg_train_loss:.4f} (Acc: {train_acc:.2f}%) | "
          f"Val Loss: {avg_val_loss:.4f} (Acc: {val_acc:.2f}%)")

    # --- 3. Early Stopping 로직 ---
    if avg_val_loss < best_val_loss:
        print(f"  🌟 검증 손실(Val Loss) 감소! ({best_val_loss:.4f} -> {avg_val_loss:.4f}). 모델을 저장합니다.")
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH) # 최고 성능 모델 저장
    else:
        patience_counter += 1
        print(f"  ⚠️ 검증 손실 증가. (Patience: {patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n🛑 {PATIENCE}번 연속으로 검증 손실이 개선되지 않아 학습을 조기 종료(Early Stopping)합니다!")
            break

print("학습 과정이 모두 끝났습니다.")

🚀 학습 및 얼리 스토핑 추적을 시작합니다...
[Epoch 1/30] Train Loss: 0.0798 (Acc: 95.17%) | Val Loss: 0.0635 (Acc: 95.45%)
  🌟 검증 손실(Val Loss) 감소! (inf -> 0.0635). 모델을 저장합니다.
[Epoch 2/30] Train Loss: 0.0658 (Acc: 95.35%) | Val Loss: 0.0690 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 1/5)
[Epoch 3/30] Train Loss: 0.0663 (Acc: 95.40%) | Val Loss: 0.0632 (Acc: 95.44%)
  🌟 검증 손실(Val Loss) 감소! (0.0635 -> 0.0632). 모델을 저장합니다.
[Epoch 4/30] Train Loss: 0.0651 (Acc: 95.40%) | Val Loss: 0.0633 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 1/5)
[Epoch 5/30] Train Loss: 0.0641 (Acc: 95.47%) | Val Loss: 0.0641 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 2/5)
[Epoch 6/30] Train Loss: 0.0675 (Acc: 95.36%) | Val Loss: 0.0634 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 3/5)
[Epoch 7/30] Train Loss: 0.0636 (Acc: 95.45%) | Val Loss: 0.0636 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 4/5)
[Epoch 8/30] Train Loss: 0.0662 (Acc: 95.45%) | Val Loss: 0.0638 (Acc: 95.45%)
  ⚠️ 검증 손실 증가. (Patience: 5/5)

🛑 5번 연속으로 검증 손실이 개선되지 않아 학습을 조기 종료(Early Stoppi

In [10]:
print("🎯 학습 중 저장된 '최고 성능의 모델'을 불러와 최종 테스트(Test)를 진행합니다...")

# 조기 종료 전 가장 성능이 좋았던 가중치(Weight) 덮어쓰기
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

test_correct, test_total = 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)

        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total
print(f"🎉 가장 똑똑한 상태 모델의 최종 실전 정확도 (Test Accuracy): {test_acc:.2f}%")

🎯 학습 중 저장된 '최고 성능의 모델'을 불러와 최종 테스트(Test)를 진행합니다...
🎉 가장 똑똑한 상태 모델의 최종 실전 정확도 (Test Accuracy): 95.45%


In [21]:
def predict_stock_chart(model, image_tensor, threshold=0.85):
    """
    확신도(Confidence)가 threshold 미만이면 강제로 Class 20(Other)으로 분류
    """
    model.eval() # 평가 모드 (필수)
    with torch.no_grad():
        image_tensor = image_tensor.to(device).unsqueeze(0) # 배치 차원 추가
        outputs = model(image_tensor)
        probabilities = F.softmax(outputs, dim=1) # 0~1 확률값 변환

        max_prob, predicted_class = torch.max(probabilities, 1)
        max_prob_value = max_prob.item()
        predicted_class_idx = predicted_class.item()

        # Threshold 방어 로직
        if max_prob_value < threshold:
            final_class = 20
            print(f"⚠️ 확률 미달 ({max_prob_value:.2f} < {threshold}): [Class 20: Other/노이즈]로 강제 분류!")
        else:
            final_class = predicted_class_idx
            print(f"✅ 확신도 통과 ({max_prob_value:.2f}): [Class {final_class}] 패턴 발견!")

        return final_class, max_prob_value

# 더미 데이터로 테스트
dummy_chart = torch.randn(1, 224, 224)
final_pred, confidence = predict_stock_chart(model, dummy_chart, threshold=0.85)

✅ 확신도 통과 (1.00): [Class 13] 패턴 발견!


In [12]:
import shutil
import os

# 코랩 임시 디스크에 저장되어 있는 모델 파일 경로
local_model_path = 'best_chart_resnet_model.pth'

# 영구 보관할 구글 드라이브 경로 (내 드라이브 최상위)
drive_model_path = '/content/drive/MyDrive/best_chart_resnet_model.pth'

# 파일 복사
if os.path.exists(local_model_path):
    shutil.copy(local_model_path, drive_model_path)
    print(f"✅ 모델 영구 저장 완료! 이제 세션이 끊겨도 안심하세요.\n저장 위치: {drive_model_path}")
else:
    print("⚠️ 오류: 로컬에 저장된 모델 파일이 없습니다. 학습(Cell 4)이 정상적으로 완료되었는지 확인하세요.")

# (선택) PC로 직접 다운로드하고 싶을 때 실행하는 코드
from google.colab import files
files.download(local_model_path)

✅ 모델 영구 저장 완료! 이제 세션이 끊겨도 안심하세요.
저장 위치: /content/drive/MyDrive/best_chart_resnet_model.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<h1>확인용</h1>

In [32]:
from PIL import Image
import torchvision.transforms as transforms

# 1. 보기 편하게 클래스 이름(라벨) 정리
class_names = {
    0: "수평 채널", 1: "상승 채널", 2: "하락 채널", 3: "대칭 삼각수렴", 4: "상승 삼각수렴", 5: "하락 삼각수렴",
    6: "하락 쐐기형", 7: "상승 쐐기형", 8: "확장형", 9: "단일 지지/저항", 10: "헤드앤숄더", 11: "역헤드앤숄더",
    12: "쌍봉", 13: "쌍바닥", 14: "삼산", 15: "삼천", 16: "컵 앤 핸들", 17: "원형 바닥", 18: "상승 깃발형",
    19: "하락 깃발형", 20: "Other(노이즈)"
}

# 2. 학습 때와 똑같은 이미지 전처리(변환) 도구 세팅
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# ==========================================
# 🎯 3. 여기서 경로를 바꿔가며 여러 이미지를 테스트해보세요!
# ==========================================
# 예시: 상승 깃발형(class_18)의 첫 번째 이미지 불러오기
test_image_path = '/content/dataset/synthetic_chart_images/class_2/img_4000.png'

# 4. 이미지 열고 전처리 적용
image = Image.open(test_image_path).convert('RGB')
image_tensor = transform(image) # 1채널 224x224 텐서로 변환됨

# 5. 방금 만든 함수에 넣어서 예측해보기!
print(f"📂 테스트 파일: {test_image_path.split('/')[-2]} / {test_image_path.split('/')[-1]}")
final_pred, confidence = predict_stock_chart(model, image_tensor, threshold=0.85)

# 6. 최종 결과 출력
print("-" * 40)
print(f"💡 AI의 최종 판정: [{class_names[final_pred]}]")
print(f"💡 확신도(Confidence): {confidence * 100:.2f}%")
print("-" * 40)

📂 테스트 파일: class_2 / img_4000.png
✅ 확신도 통과 (1.00): [Class 12] 패턴 발견!
----------------------------------------
💡 AI의 최종 판정: [쌍봉]
💡 확신도(Confidence): 100.00%
----------------------------------------
